# X-Ray Fine-tune DenseNet121 Model Notebook
Đây là notebook tự động chạy fine-tuning mô hình dựa trên tập dữ liệu bác sĩ đã gán nhãn được đồng bộ từ local.

In [ ]:
# Cell 1: Cài đặt các thư viện cần thiết
!pip install -q torchxrayvision mlflow boto3 scikit-learn pandas tqdm

In [ ]:
# Cell 2: Khai báo thư viện & cấu hình MinIO connection
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchxrayvision as xrv
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import boto3
from botocore.client import Config
from PIL import Image
from tqdm import tqdm

# Các cấu hình này sẽ đọc trực tiếp từ môi trường Kaggle Secrets hoặc truyền qua ENV
# Vì chạy trên Kaggle Cloud, bạn cần EXPOSE port 9000 của MinIO ra internet bằng ngrok và cấu hình bên dưới:
MINIO_ENDPOINT = os.getenv("MINIO_S3_PUBLIC_URL", "https://your-ngrok-url.ngrok-free.app")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")
BUCKET_MODELS = "xray-models"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
)
print("MinIO client initialized.")

In [ ]:
# Cell 3: Đọc dataset từ Kaggle input
# Tìm thư mục input
input_dirs = [os.path.join("/kaggle/input", d) for d in os.listdir("/kaggle/input")]
dataset_dir = input_dirs[0] if input_dirs else None

if dataset_dir:
    # Giải nén file zip nếu có
    zip_file = [os.path.join(dataset_dir, f) for f in os.listdir(dataset_dir) if f.endswith(".zip")]
    if zip_file:
        import zipfile
        with zipfile.ZipFile(zip_file[0], 'r') as zip_ref:
            zip_ref.extractall("/kaggle/working/data")
        data_root = "/kaggle/working/data"
    else:
        data_root = dataset_dir
else:
    raise Exception("No input dataset found on Kaggle!")

df = pd.read_csv(os.path.join(data_root, "labels.csv"))
print(f"Loaded {len(df)} images details from labels.csv")
df.head()

In [ ]:
# Cell 4: Tạo Dataset Loader
class ChestXRayDataset(Dataset):
    def __init__(self, df, img_dir, pathologies, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.pathologies = pathologies
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["filename"]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Load ảnh dưới dạng grayscale & convert thành numpy array chuẩn
        img = Image.open(img_path).convert("L")
        img = np.array(img)
        
        # Normalize về [-1024, 1024] theo chuẩn TorchXRayVision
        img = xrv.datasets.normalize(img, 255)
        
        # Resize về 224x224
        img = np.array(Image.fromarray(img).resize((224, 224)))
        
        # Thêm channel dimension
        img = img[None, :, :]
        img = torch.from_numpy(img).float()
        
        # Lấy các cột label
        labels = row[self.pathologies].values.astype(np.float32)
        return img, torch.from_numpy(labels)

In [ ]:
# Cell 5: Load Model & Fine-tune weights
# Kiểm tra xem đã có model.pt cũ lưu trên MinIO chưa để tiếp tục train
model_local_path = "/kaggle/working/base_model.pt"
has_checkpoint = False
try:
    # Thử download model.pt gần nhất
    s3.download_file(BUCKET_MODELS, "latest_model.pt", model_local_path)
    has_checkpoint = True
    print("Successfully downloaded latest checkpoint from MinIO.")
except Exception as e:
    print("No checkpoint found. Training from pretrained TorchXRayVision base weights.")

model = xrv.models.DenseNet(weights="densenet121-res224-all")
if has_checkpoint:
    try:
        model.load_state_dict(torch.load(model_local_path))
        print("Loaded checkpoint weights.")
    except Exception as e:
        print(f"Failed to load checkpoint: {e}. Reverting to base.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

In [ ]:
# Cell 6: Huấn luyện mô hình
pathologies = model.pathologies
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = ChestXRayDataset(train_df, os.path.join(data_root, "images"), pathologies)
val_dataset = ChestXRayDataset(val_df, os.path.join(data_root, "images"), pathologies)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

criterion = nn.BCEWithLogitsLoss()

# Chiến lược đóng băng: nếu ít dữ liệu (<500 ảnh) thì đóng băng các lớp đầu
if len(df) < 500:
    print("Data is limited (< 500 scans). Freezing feature extractor, only training classifier.")
    for param in model.features.parameters():
        param.requires_grad = False
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
else:
    print("Sufficient data available. Fine-tuning entire network.")
    optimizer = optim.Adam(model.parameters(), lr=1e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# Chạy train loop 3 epoch làm demo
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Validation & AUC Calculation
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Tính AUC trung bình
    aucs = []
    per_disease_auc = {}
    for idx, p in enumerate(pathologies):
        try:
            auc = roc_auc_score(all_labels[:, idx], all_preds[:, idx])
            aucs.append(auc)
            per_disease_auc[p] = float(auc)
        except ValueError:
            # Bỏ qua nếu cột chỉ chứa 1 class trong val set
            pass
            
    avg_auc = np.mean(aucs) if aucs else 0.5
    print(f"Epoch {epoch+1} done. Loss: {total_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Avg AUC: {avg_auc:.4f}")
    scheduler.step(val_loss)


In [ ]:
# Cell 7: Upload Model và Metrics đã được sinh ra ngược về MinIO
import time
timestamp = int(time.time())

# Lưu model checkpoint local
model_save_path = f"/kaggle/working/model_v{timestamp}.pt"
metrics_save_path = f"/kaggle/working/metrics_v{timestamp}.json"

torch.save(model.state_dict(), model_save_path)

metrics_data = {
    "avg_auc": float(avg_auc),
    "per_disease_auc": per_disease_auc
}
with open(metrics_save_path, "w") as f:
    json.dump(metrics_data, f)
    
print("Uploading model.pt and metrics.json back to MinIO...")
try:
    # Upload checkpoint chính để tiếp tục fine-tune vòng sau
    s3.upload_file(model_save_path, BUCKET_MODELS, "latest_model.pt")
    # Upload phiên bản lưu trữ theo timestamp
    s3.upload_file(model_save_path, BUCKET_MODELS, f"model_v{timestamp}.pt")
    s3.upload_file(metrics_save_path, BUCKET_MODELS, f"metrics_v{timestamp}.json")
    print("Upload successfully completed.")
except Exception as e:
    print(f"Upload to MinIO failed: {e}")